#  SILVER LAYER

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS ecommerce.silver_raw
""")

DataFrame[]

In [0]:
# ============================================================
# 06_SILVER
#
# Silver Layer Data Cleaning and Standardization
#
# Input:
#   ecommerce.silver
#
# Output:
#   ecommerce.silver
#
# Responsibilities:
#   - Remove unnecessary columns
#   - Fix data types
#   - Clean text fields
#   - Remove duplicates
#   - Handle invalid records
#
# CDC has already happened before this notebook.
# ============================================================


from pyspark.sql.functions import *



# ============================================================
# Configuration
# ============================================================

catalog = "ecommerce"

schema = "silver"



# ============================================================
# 1. CUSTOMERS CLEANING
# ============================================================


customers = spark.table(
    f"{catalog}.{schema}.customers"
)



customers_clean = (

    customers


    # Remove CDC metadata if present

    .drop(
        "operation",
        "operation_timestamp",
        "_rescued_data"
    )


    # Remove records without primary key

    .filter(
        col("customer_id").isNotNull()
    )


    # Remove duplicates

    .dropDuplicates(
        ["customer_id"]
    )


    # Data type conversion

    .withColumn(
        "customer_id",
        col("customer_id").cast("int")
    )


    .withColumn(
        "loyalty_points",
        col("loyalty_points").cast("int")
    )


    .withColumn(
        "date_of_birth",
        to_date(col("date_of_birth"))
    )


    .withColumn(
        "registration_date",
        to_date(col("registration_date"))
    )


    # String cleaning

    .withColumn(
        "first_name",
        initcap(trim(col("first_name")))
    )


    .withColumn(
        "last_name",
        initcap(trim(col("last_name")))
    )


    .withColumn(
        "city",
        initcap(trim(col("city")))
    )


    .withColumn(
        "country",
        upper(trim(col("country")))
    )


    # Handle missing values

    .fillna(
        {
            "membership":"UNKNOWN",
            "status":"UNKNOWN"
        }
    )

)



customers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "ecommerce.silver.customers"
    )


print("Customers Silver completed")





# ============================================================
# 2. PRODUCTS CLEANING
# ============================================================


products = spark.table(
    f"{catalog}.{schema}.products"
)



products_clean = (

    products


    .drop(
        "operation",
        "operation_timestamp",
        "_rescued_data"
    )


    .filter(
        col("product_id").isNotNull()
    )


    .dropDuplicates(
        ["product_id"]
    )


    .withColumn(
        "product_id",
        col("product_id").cast("int")
    )


    .withColumn(
        "price",
        col("price").cast("decimal(10,2)")
    )


    .withColumn(
        "cost_price",
        col("cost_price").cast("decimal(10,2)")
    )


    .withColumn(
        "stock_quantity",
        col("stock_quantity").cast("int")
    )


    .withColumn(
        "rating",
        col("rating").cast("double")
    )


    .withColumn(
        "product_name",
        trim(col("product_name"))
    )

)



products_clean.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    "ecommerce.silver.products"
)



print("Products Silver completed")





# ============================================================
# 3. ORDERS CLEANING
# ============================================================


orders = spark.table(
    f"{catalog}.{schema}.orders"
)



# First, get valid customer and product IDs for referential integrity
valid_customer_ids = customers_clean.select("customer_id").distinct()
valid_product_ids = products_clean.select("product_id").distinct()


orders_clean = (

    orders


    .drop(
        "operation",
        "operation_timestamp",
        "_rescued_data"
    )


    .filter(
        col("order_id").isNotNull()
    )


    .dropDuplicates(
        ["order_id"]
    )


    # Filter out orders with invalid foreign keys
    .join(
        valid_customer_ids,
        "customer_id",
        "inner"
    )


    .join(
        valid_product_ids,
        "product_id",
        "inner"
    )


    .withColumn(
        "order_id",
        col("order_id").cast("int")
    )


    .withColumn(
        "customer_id",
        col("customer_id").cast("int")
    )


    .withColumn(
        "product_id",
        col("product_id").cast("int")
    )


    .withColumn(
        "quantity",
        col("quantity").cast("int")
    )


    .withColumn(
        "unit_price",
        col("unit_price").cast("decimal(10,2)")
    )


    .withColumn(
        "total_amount",
        col("total_amount").cast("decimal(10,2)")
    )


    .withColumn(
        "discount",
        col("discount").cast("decimal(10,2)")
    )


    .withColumn(
        "tax",
        col("tax").cast("decimal(10,2)")
    )


    .withColumn(
        "shipping_cost",
        col("shipping_cost").cast("decimal(10,2)")
    )


    .withColumn(
        "order_date",
        to_date(col("order_date"))
    )


    .withColumn(
        "city",
        initcap(trim(col("city")))
    )

)



orders_clean.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    "ecommerce.silver.orders"
)



print("Orders Silver completed")





# ============================================================
# 4. PAYMENTS CLEANING
# ============================================================


payments = spark.table(
    f"{catalog}.{schema}.payments"
)



payments_clean = (

    payments


    .drop(
        "operation",
        "operation_timestamp",
        "_rescued_data"
    )


    .filter(
        col("payment_id").isNotNull()
    )


    .dropDuplicates(
        ["payment_id"]
    )


    .withColumn(
        "payment_id",
        col("payment_id").cast("int")
    )


    .withColumn(
        "order_id",
        col("order_id").cast("int")
    )


    .withColumn(
        "customer_id",
        col("customer_id").cast("int")
    )


    .withColumn(
        "amount",
        col("amount").cast("decimal(10,2)")
    )


    .withColumn(
        "payment_date",
        to_date(col("payment_date"))
    )


)



payments_clean.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    "ecommerce.silver.payments"
)



print("Payments Silver completed")





# ============================================================
# 5. RETURNS CLEANING
# ============================================================


returns = spark.table(
    f"{catalog}.{schema}.returns"
)



returns_clean = (

    returns


    .drop(
        "operation",
        "operation_timestamp",
        "_rescued_data"
    )


    .filter(
        col("return_id").isNotNull()
    )


    .dropDuplicates(
        ["return_id"]
    )


    .withColumn(
        "return_id",
        col("return_id").cast("int")
    )


    .withColumn(
        "order_id",
        col("order_id").cast("int")
    )


    .withColumn(
        "customer_id",
        col("customer_id").cast("int")
    )


    .withColumn(
        "product_id",
        col("product_id").cast("int")
    )


    .withColumn(
        "refund_amount",
        col("refund_amount").cast("decimal(10,2)")
    )


    .withColumn(
        "return_date",
        to_date(col("return_date"))
    )


)



returns_clean.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable(
    "ecommerce.silver.returns"
)



print("Returns Silver completed")





# ============================================================
# Final Validation
# ============================================================


print("Silver Tables:")

spark.sql("""
SHOW TABLES IN ecommerce.silver
""").show()



for table in [
    "customers",
    "products",
    "orders",
    "payments",
    "returns"
]:

    print("\n")
    print("="*50)
    print(table.upper())
    print("="*50)

    display(
        spark.table(
            f"ecommerce.silver.{table}"
        ).limit(10)
    )

Customers Silver completed
Products Silver completed
Orders Silver completed
Payments Silver completed
Returns Silver completed
Silver Tables:
+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|  silver|customers|      false|
|  silver|   orders|      false|
|  silver| payments|      false|
|  silver| products|      false|
|  silver|  returns|      false|
+--------+---------+-----------+



CUSTOMERS


city,country,customer_id,date_of_birth,email,first_name,gender,last_name,loyalty_points,membership,phone,postal_code,registration_date,state,status
Karachi,PAKISTAN,12,1984-10-15,william.gray12@email.com,William,Male,Gray,3126,Silver,+923034106178,60432,2025-05-14,Sindh,Active
Karachi,PAKISTAN,18,2005-03-05,christine.adams18@email.com,Christine,Female,Adams,3059,Bronze,+923360609191,95180,2024-09-30,Sindh,Active
Mardan,PAKISTAN,38,1964-12-31,dennis.mason38@email.com,Dennis,Male,Mason,1450,Gold,+923464181650,31299,2023-11-13,KPK,Active
Quetta,PAKISTAN,67,2006-11-21,yvonne.huff67@email.com,Yvonne,Female,Huff,2011,Gold,+923144093758,52245,2025-06-11,Balochistan,Active
Rawalpindi,PAKISTAN,70,2007-10-03,amy.gray70@email.com,Amy,Female,Gray,4002,Silver,+923386587142,63271,2023-06-25,Punjab,Active
Quetta,PAKISTAN,93,1969-12-19,ashley.tran93@email.com,Ashley,Female,Tran,1184,Bronze,+923245235368,82255,2025-08-25,Balochistan,Inactive
Peshawar,PAKISTAN,161,1974-03-05,james.smith161@email.com,James,Male,Smith,1207,Gold,+923241226417,77286,2023-11-25,KPK,Active
Hyderabad,PAKISTAN,186,2007-09-05,kyle.martinez186@email.com,Kyle,Male,Martinez,4238,Bronze,+923250092294,99143,2023-06-01,Sindh,Active
Hyderabad,PAKISTAN,190,1999-06-15,tyler.reid190@email.com,Tyler,Male,Reid,665,Bronze,+923443045345,13718,2026-02-28,Sindh,Inactive
Karachi,PAKISTAN,218,1980-11-07,amber.everett218@email.com,Amber,Female,Everett,1582,Bronze,+923304825222,98395,2024-11-21,Sindh,Active




PRODUCTS


brand,category,cost_price,launch_date,price,product_id,product_name,rating,status,stock_quantity,supplier
Maybelline,Beauty,546.18,2024-06-23,873.16,12,Maybelline Sit 241,4.0,Active,1043,Maybelline Supplier
Apple,Electronics,1318.69,2024-01-26,1998.91,18,Apple Almost 414,3.1,Active,490,Apple Supplier
Under Armour,Sports,16.70,2025-12-06,25.19,38,Under Armour True 406,4.3,Discontinued,1908,Under Armour Supplier
Tefal,Home,1280.90,2024-05-16,2019.61,67,Tefal Human 529,4.5,Active,1102,Tefal Supplier
Under Armour,Sports,130.50,2020-04-21,171.52,70,Under Armour Finally 987,3.4,Active,945,Under Armour Supplier
H&M,Fashion,678.52,2024-04-07,896.57,93,H&M Long 980,5.0,Active,1205,H&M Supplier
IKEA,Home,748.59,2020-02-01,984.64,161,IKEA Pattern 232,4.1,Active,778,IKEA Supplier
L'Oreal,Beauty,1225.97,2022-06-28,1890.83,186,L'Oreal Pick 127,3.1,Active,849,L'Oreal Supplier
Under Armour,Sports,171.22,2024-09-29,213.01,190,Under Armour On 408,3.1,Active,249,Under Armour Supplier
HP,Electronics,889.75,2021-11-21,1237.04,218,HP Movement 275,4.2,Discontinued,1349,HP Supplier




ORDERS


product_id,customer_id,city,country,delivered_date,discount,order_date,order_id,order_status,payment_id,quantity,shipped_date,shipping_cost,tax,total_amount,unit_price
4665,241,Quetta,Pakistan,2026-03-17T14:04:51,0.15,2026-03-10,10002,Shipped,10002,2,2026-03-12T14:04:51,8.43,103.15,988.34,515.74
458,583,Quetta,Pakistan,2025-04-24T06:25:54,0.00,2025-04-14,10022,Delivered,10022,4,2025-04-18T06:25:54,16.62,548.39,6048.89,1370.97
2415,819,Quetta,Pakistan,2025-10-05T20:19:16,0.20,2025-09-30,1003,Shipped,1003,4,2025-10-02T20:19:16,48.50,546.02,4962.65,1365.04
2661,868,Islamabad,Pakistan,2024-10-22T12:57:47,0.10,2024-10-13,10045,Shipped,10045,5,2024-10-18T12:57:47,38.57,64.89,687.51,129.79
2022,964,Islamabad,Pakistan,2025-10-22T23:45:04,0.00,2025-10-14,10060,Pending,10060,4,2025-10-16T23:45:04,11.43,443.69,4892.04,1109.23
465,407,Karachi,Pakistan,2025-09-09T00:46:17,0.20,2025-08-31,10091,Delivered,10091,3,2025-09-05T00:46:17,48.22,456.41,4155.92,1521.37
3889,493,Multan,Pakistan,2025-05-16T15:58:48,0.00,2025-05-10,10103,Delivered,10103,4,2025-05-13T15:58:48,45.44,498.58,5529.86,1246.46
2718,912,Islamabad,Pakistan,2026-01-17T06:47:24,0.10,2026-01-06,10158,Shipped,10158,3,2026-01-11T06:47:24,36.05,425.12,4287.29,1417.08
2635,357,Multan,Pakistan,2026-04-05T15:48:45,0.15,2026-03-24,10192,Shipped,10192,4,2026-03-29T15:48:45,28.37,587.52,5609.78,1468.79
2267,108,Peshawar,Pakistan,2025-04-07T08:17:53,0.10,2025-03-30,10197,Delivered,10197,4,2025-04-01T08:17:53,20.89,54.86,569.53,137.16




PAYMENTS


amount,currency,customer_id,order_id,payment_date,payment_id,payment_method,payment_status,transaction_reference
864.06,PKR,612,10002,2026-02-20,10002,Easypaisa,Failed,TXN100010002
9404.90,PKR,749,10022,2025-06-07,10022,PayPal,Pending,TXN100010022
6613.42,PKR,539,1003,2024-09-19,1003,PayPal,Completed,TXN100001003
6523.17,PKR,370,10045,2026-02-09,10045,Easypaisa,Completed,TXN100010045
9656.15,PKR,326,10060,2024-07-25,10060,Bank Transfer,Completed,TXN100010060
1344.43,PKR,388,10091,2024-12-19,10091,JazzCash,Completed,TXN100010091
5023.51,PKR,79,10103,2026-04-07,10103,JazzCash,Completed,TXN100010103
2491.14,PKR,349,10158,2025-10-16,10158,Debit Card,Completed,TXN100010158
8656.63,PKR,69,10192,2025-10-18,10192,Credit Card,Completed,TXN100010192
2281.68,PKR,722,10197,2024-02-02,10197,Debit Card,Completed,TXN100010197




RETURNS


customer_id,order_id,product_id,refund_amount,return_date,return_id,return_reason,return_status
411,51749,4838,8935.03,2024-06-21,1003,Damaged,Refunded
174,44250,162,4663.00,2026-06-03,1021,Defective,Approved
903,23672,153,5810.75,2024-08-06,1030,Defective,Refunded
384,69923,926,4801.06,2024-09-09,1044,Late Delivery,Approved
457,64176,3180,6043.37,2024-04-02,1088,Changed Mind,Approved
701,16750,2461,3025.27,2024-02-18,1092,Wrong Product,Approved
606,67287,3997,5330.26,2025-03-08,1108,Wrong Product,Refunded
860,92487,4748,5382.08,2024-11-24,1160,Changed Mind,Pending
29,12679,3932,5966.05,2025-10-03,1195,Changed Mind,Refunded
842,30022,263,755.27,2025-08-03,12,Late Delivery,Approved


## Silver Schema evaluation

In [0]:
catalog = "ecommerce"
silver_schema = "silver"

datasets = [
    "customers",
    "products",
    "orders",
    "payments",
    "returns"
]
for dataset in datasets:

    print("\n")
    print("=" * 70)
    print(dataset.upper())
    print("=" * 70)

    spark.table(
        f"{catalog}.{silver_schema}.{dataset}"
    ).printSchema()



CUSTOMERS
root
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- email: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- loyalty_points: integer (nullable = true)
 |-- membership: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- status: string (nullable = true)



PRODUCTS
root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cost_price: decimal(10,2) (nullable = true)
 |-- launch_date: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- status: stri